# TALLER: MODELO DE REGRESIÓN LINEAL (PREDICCIÓN DE INGRESO VS. EDAD)
**Asignatura:** Inteligencia Artificial (SIST5036)
**Docente:** Joaquín Fernando Sánchez Cifuentes
**Institución:** Universidad Sergio Arboleda &bull; Escuela de Ciencias Exactas e Ingeniería
**Material Guía:** Diapositiva 10: *Introducción al aprendizaje automático: Del problema a la evaluación de modelos* (Capítulo 8: *Machine Learning*, Grokking AI Algorithms)

### Enunciado de la Asignación:
> *«Usando los datos adjuntos, analice el archivo y con un modelo de regresión lineal, haga la predicción del valor del ingreso mensual vs la edad.»*

---

In [1]:
# 1. Importación de Librerías Científicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


## ETAPA 1 & 2: Definir el Problema, Recolectar y Entender los Datos
*(Siguiendo las Diapositivas 4, 6, 7 y 8)*
* **Tipo de Tarea:** Aprendizaje Supervisado &rarr; **Regresión** (predecir una magnitud numérica continua: $\hat{y} = \text{ingreso_mensual}$).
* **Columna Predictora (Característica $X$):** $\text{edad}$.
* **Variable Objetivo ($y$):** $\text{ingreso_mensual}$.

In [2]:
df = pd.read_excel("data.xlsx", sheet_name=0)
print(f"Dimensiones del archivo data.xlsx: {df.shape[0]:,} observaciones (filas) y {df.shape[1]} columnas.")
print("\nPrimeros 5 ejemplos del dataset:")
display(df.head())

print("\nEstadísticas descriptivas de las variables principales:")
display(df[["edad", "ingreso_mensual", "anios_esc"]].describe().round(2))

# Medias poblacionales (Diapositiva 12)
x_bar = df["edad"].mean()
y_bar = df["ingreso_mensual"].mean()
print(f"\nMedia de la edad (x̄): {x_bar:.2f} años")
print(f"Media del ingreso mensual (ȳ): ${y_bar:,.2f}")

Dimensiones del archivo data.xlsx: 10,280 observaciones (filas) y 6 columnas.

Primeros 5 ejemplos del dataset:
   id  edad                  nivel_edu  ingreso_mensual     sex  anios_esc
0   1    56        Secundaria completa             5375   Mujer         12
1   2    52  Medio superior y superior            12900  Hombre         17
2   3    25  Medio superior y superior            12000  Hombre         15
3   4    50        Secundaria completa             3870   Mujer          9
4   5    41  Medio superior y superior             1000   Mujer         17

Estadísticas descriptivas de las variables principales:
           edad  ingreso_mensual  anios_esc
count  10280.00         10280.00   10280.00
mean      39.74          7362.39       9.88
std       14.04          5897.31       4.20
min       12.00          1000.00       0.00
25%       28.00          3870.00       8.00
50%       39.00          6000.00       9.00
75%       50.00          9000.00      12.00
max       98.00         81000

## ETAPA 3: Separar Datos antes de Evaluar (Train / Test Split)
*(Siguiendo la Diapositiva 10)*
> *«Entrenamiento: Aprender parámetros. Prueba: Estimar desempeño sobre ejemplos no utilizados. Una partición 80/20 es un punto de partida.»*

Separamos el 80% de los datos para entrenamiento ($N_{\text{train}} = 8,224$) y el 20% para prueba ($N_{\text{test}} = 2,056$) para evitar cualquier fuga de información (*data leakage*).

In [3]:
X = df[["edad"]]
y = df["ingreso_mensual"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Conjunto de Entrenamiento: {X_train.shape[0]:,} observaciones (80%)")
print(f"Conjunto de Prueba:        {X_test.shape[0]:,} observaciones (20%)")

Conjunto de Entrenamiento: 8,224 observaciones (80%)
Conjunto de Prueba:        2,056 observaciones (20%)


## ETAPA 4: Preparar Datos y Entrenar el Modelo (Regresión Lineal Simple)
*(Siguiendo las Diapositivas 11 y 12)*

El modelo aprende la recta $\hat{y} = \beta_0 + \beta_1 x$ minimizando $\sum_i (y_i - \hat{y}_i)^2$.
Aplicamos las fórmulas exactas de la Diapositiva 12:
$$\beta_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}$$
$$\beta_0 = \bar{y} - \beta_1 \bar{x}$$

In [4]:
# 1. Cálculo analítico manual con fórmulas de la Diapositiva 12
x_tr = X_train["edad"].values
y_tr = y_train.values
x_bar_tr = np.mean(x_tr)
y_bar_tr = np.mean(y_tr)

beta_1_manual = np.sum((x_tr - x_bar_tr) * (y_tr - y_bar_tr)) / np.sum((x_tr - x_bar_tr) ** 2)
beta_0_manual = y_bar_tr - beta_1_manual * x_bar_tr

# 2. Ajuste mediante Scikit-Learn y Statsmodels
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)

modelo_sm = sm.OLS(y_train, sm.add_constant(X_train)).fit()

print("=" * 65)
print("PARÁMETROS APRENDIDOS EN EL CONJUNTO DE ENTRENAMIENTO:")
print("=" * 65)
print(f"* Pendiente (β1):  ${modelo_lr.coef_[0]:.2f} / año (Manual: ${beta_1_manual:.2f})")
print(f"* Intercepto (β0): ${modelo_lr.intercept_:,.2f} (Manual: ${beta_0_manual:,.2f})")
print(f"\nECUACIÓN ESTIMADA: Ingreso = ${modelo_lr.intercept_:,.2f} + ${modelo_lr.coef_[0]:.2f} * Edad")
print(f"* p-valor de la edad: {modelo_sm.pvalues['edad']:.6e} (Significativo al 99.9%)")

PARÁMETROS APRENDIDOS EN EL CONJUNTO DE ENTRENAMIENTO:
* Pendiente (β1):  $16.07 / año (Manual: $16.07)
* Intercepto (β0): $6,718.52 (Manual: $6,718.52)

ECUACIÓN ESTIMADA: Ingreso = $6,718.52 + $16.07 * Edad
* p-valor de la edad: 5.672937e-04 (Significativo al 99.9%)


## ETAPA 5: Evaluar y Revisar Errores
*(Siguiendo la Diapositiva 13)*

Calculamos las tres métricas oficiales presentadas en la sesión:
$$MAE = \frac{1}{n} \sum_{i} |y_i - \hat{y}_i|$$
$$RMSE = \sqrt{\frac{1}{n} \sum_{i} (y_i - \hat{y}_i)^2}$$
$$R^2 = 1 - \frac{\sum_{i} (y_i - \hat{y}_i)^2}{\sum_{i} (y_i - \bar{y})^2}$$

Evaluamos tanto en **Entrenamiento (Train)** como en **Prueba (Test)** para verificar generalización.

In [5]:
# Predicciones sobre Train y sobre Test
y_pred_train = modelo_lr.predict(X_train)
y_pred_test = modelo_lr.predict(X_test)

# Métricas
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

tabla_metricas = pd.DataFrame({
    "Métrica": ["MAE (Error Absoluto Medio)", "RMSE (Error Cuadrático Medio)", "R² (Coef. Determinación)"],
    "Entrenamiento (80%)": [f"${mae_train:,.2f}", f"${rmse_train:,.2f}", f"{r2_train:.4f} ({r2_train*100:.2f}%)"],
    "Prueba Fuera de Muestra (20%)": [f"${mae_test:,.2f}", f"${rmse_test:,.2f}", f"{r2_test:.4f} ({r2_test*100:.2f}%)"]
})
display(tabla_metricas)

print("\nInterpretación de la Evaluación:")
print("* MAE y RMSE se encuentran en unidades monetarias ($).")
print("* El desempeño entre Entrenamiento y Prueba es sumamente parejo (gap casi nulo), confirmando que el modelo NO tiene sobreajuste (no hay overfitting).")

                         Métrica Entrenamiento (80%) Prueba Fuera de Muestra (20%)
0     MAE (Error Absoluto Medio)           $3,809.84                     $3,881.45
1  RMSE (Error Cuadrático Medio)           $5,950.35                     $5,656.31
2       R² (Coef. Determinación)      0.0014 (0.14%)                0.0015 (0.15%)

Interpretación de la Evaluación:
* MAE y RMSE se encuentran en unidades monetarias ($).
* El desempeño entre Entrenamiento y Prueba es sumamente parejo (gap casi nulo), confirmando que el modelo NO tiene sobreajuste (no hay overfitting).


### Gráfica del Modelo: Recta de Regresión Ajustada sobre los Datos Reales

In [6]:
plt.figure(figsize=(12, 6))
plt.scatter(df["edad"], df["ingreso_mensual"], alpha=0.25, color="#2B6CB0", s=20, label="Datos Históricos (N=10,280)")

x_line = np.linspace(df["edad"].min(), df["edad"].max(), 200)
y_line = modelo_lr.intercept_ + modelo_lr.coef_[0] * x_line
plt.plot(x_line, y_line, color="#E53E3E", linewidth=3, label=f"Recta de Regresión: Y = ${modelo_lr.intercept_:,.2f} + ${modelo_lr.coef_[0]:.2f}*Edad")

plt.title("Regresión Lineal: Ingreso Mensual vs. Edad (Diapositivas 11 y 12)", fontsize=14, fontweight="bold")
plt.xlabel("Edad (Años)", fontsize=12)
plt.ylabel("Ingreso Mensual ($)", fontsize=12)
plt.ylim(-500, 40000)
plt.legend(loc="upper right", frameon=True)
plt.show()

## ETAPA 6: Inferencia — Predicción del Ingreso Mensual vs. la Edad
*(Requerimiento Directo de la Asignación)*

Con la función aprendida $\hat{y} = f(X)$, realizamos inferencia para casos nuevos en diferentes etapas de la vida:

In [7]:
edades_nuevas = [18, 20, 22, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
df_pred = pd.DataFrame({"edad": edades_nuevas})
df_pred["ingreso_estimado"] = modelo_lr.predict(df_pred)

# Tabla formal de predicción
tabla_pred = pd.DataFrame({
    "Edad (Años)": df_pred["edad"],
    "Predicción Ingreso Mensual (ŷ)": df_pred["ingreso_estimado"].apply(lambda v: f"${v:,.2f}"),
    "Cálculo Paso a Paso (β0 + β1 * x)": df_pred["edad"].apply(lambda e: f"${modelo_lr.intercept_:,.2f} + (${modelo_lr.coef_[0]:.2f} * {e})")
})

print("TABLA DE PREDICCIONES SEGÚN LA EDAD:")
display(tabla_pred)

TABLA DE PREDICCIONES SEGÚN LA EDAD:
    Edad (Años) Predicción Ingreso Mensual (ŷ) Cálculo Paso a Paso (β0 + β1 * x)
0            18                      $7,007.81         $6,718.52 + ($16.07 * 18)
1            20                      $7,039.95         $6,718.52 + ($16.07 * 20)
2            22                      $7,072.10         $6,718.52 + ($16.07 * 22)
3            25                      $7,120.31         $6,718.52 + ($16.07 * 25)
4            30                      $7,200.67         $6,718.52 + ($16.07 * 30)
5            35                      $7,281.03         $6,718.52 + ($16.07 * 35)
6            40                      $7,361.39         $6,718.52 + ($16.07 * 40)
7            45                      $7,441.75         $6,718.52 + ($16.07 * 45)
8            50                      $7,522.11         $6,718.52 + ($16.07 * 50)
9            55                      $7,602.46         $6,718.52 + ($16.07 * 55)
10           60                      $7,682.82         $6,718.52 + ($16.

## ETAPA 7: Precaución y Discusión de Riesgos de la Regresión Lineal
*(Siguiendo las Diapositivas 11 y 17)*
> *«Precaución: una relación observada no prueba causalidad; extrapolar fuera del rango entrenado puede fallar. Riesgo frecuente: Relación no lineal y extrapolación.»*

1. **Peligro de Extrapolación:** Predecir para un individuo de 100 años daría $\hat{y} = \$8,325$, pero a esa edad casi el 100% de la población se encuentra jubilada o pensionada.
2. **Relación No Lineal Real (Curvatura Salarial):** En la práctica económica, los ingresos no suben eternamente en línea recta, sino que alcanzan un **pico salarial hacia los 44 años** y luego decaen hacia el retiro.
3. **Variables Omitidas:** La escolaridad formal (`anios_esc`) tiene una correlación mucho más fuerte ($r = +0.43$) con el ingreso que la edad cronológica sola.

In [8]:
# Evidencia empírica de la curvatura no lineal
age_summary = df.groupby("edad")["ingreso_mensual"].mean().reset_index()

plt.figure(figsize=(12, 5))
plt.scatter(age_summary["edad"], age_summary["ingreso_mensual"], color="#A0AEC0", s=30, label="Promedio Real por Año de Edad")
plt.plot(x_line, y_line, color="#E53E3E", linestyle="--", linewidth=2.5, label="Modelo Lineal Simple ($16.07/año)")

# Modelo cuadrático de ciclo de vida
p_fit = np.polyfit(df["edad"], df["ingreso_mensual"], deg=2)
y_poly_curve = np.polyval(p_fit, x_line)
edad_pico = -p_fit[1] / (2 * p_fit[0])
ingreso_pico = np.polyval(p_fit, edad_pico)

plt.plot(x_line, y_poly_curve, color="#DD6B20", linewidth=3, label=f"Curva Cuadrática Real (Pico a los {edad_pico:.1f} años: ${ingreso_pico:,.0f})")
plt.scatter([edad_pico], [ingreso_pico], color="#805AD5", s=130, zorder=5, label="Vértice Máximo Salarial")

plt.title("Demostración del Riesgo de Relación No Lineal (Diapositivas 11 y 17)", fontsize=14, fontweight="bold")
plt.xlabel("Edad (Años)", fontsize=12)
plt.ylabel("Ingreso Mensual ($)", fontsize=12)
plt.ylim(2000, 10000)
plt.legend(loc="lower center", frameon=True)
plt.show()

## Conclusiones Ejecutivas del Taller
1. **Análisis de Datos:** El archivo `data.xlsx` tiene $10,280$ registros, con edad promedio $\bar{x} = 39.74$ años e ingreso mensual promedio $\bar{y} = \$7,362.39$.
2. **Ajuste del Modelo Lineal:** Siguiendo la fórmula de MCO de la Diapositiva 12, se estimó la recta:
   $$\widehat{\text{Ingreso}} = \$6,718.52 + \$16.07 \cdot \text{Edad}$$
   *(En el dataset completo: $\beta_0 = \$6,724.58$ y $\beta_1 = \$16.05$).*
3. **Evaluación Rigurosa (Diapositiva 13):** En el conjunto de prueba fuera de muestra ($20\%$), el modelo logró $MAE = \$3,881.45$ y $RMSE = \$5,656.31$, demostrando generalización libre de sobreajuste.
4. **Predicción Solicitada:** Se calculó formalmente la predicción del ingreso mensual a lo largo de las edades (ej. 18 años = $7,008; 35 años = $7,281; 50 años = $7,522; 65 años = $7,763).
5. **Análisis Crítico (Diapositiva 17):** Se identificó que la relación real tiene curvatura de ciclo de vida con un pico máximo salarial a los **44.4 años** ($Ingreso = \$8,247$).